[![OpenSD2026 - Belgium](../../assets/OpenSD_bannerlogo.jpg)](https://www.vub.be/en/event/opensd-summer-school-belgium)

Copyright © 2026 OpenSD2026 contributors. All rights reserved. The materials in this repository are provided for educational use only and may not be copied, redistributed, or modified without prior permission. See [LICENSE](../../LICENSE).

# Part I of the WS4

In this first part of WS4 we will work on the classic task of **Data-normalization** through regression. **Data-normalization** is a key part of any Structural Health Monitoring strategy and serves to differentiate the reversible (and environmental or operational) variability seen in measurement data. A classic example is how many measurements depend on Temperature. This change in temperature is not a problem, but might hide more problematic changes in the measurement quantity. Data-normalization in that example serves to remove that temperature dependency.

In this WS we will use the modelling of the Power Curve of an operational wind turbine to demonstrate this practice.

# Modelling the Power Curve of a Wind Turbine [Estimated time: 1H30M]

In this notebook, we use collected SCADA data to model the power curve of an operational wind turbine. A power curve describes the relationship between wind speed and the power produced by the turbine, i.e. P(Ws). Aside from the Windspeed, operators can run the turbine in different modes, or **states**.  The relationship between power and windspeed thus depends on the turbine's state and can therefore vary even within similar environmental conditions.

In this workshop we will use a neural network to learn the mapping from wind speed, and later other relevant variables, to the generated power.

The goal is to understand the modelling process rather than to present the most sophisticated model. This problem can also be approached with simpler methods that are easy to implement, such as binning techniques.

# Dataset

The dataset comes from the Aventa AV-7 (6 kW) IET-OST Research Wind Turbine SCADA system.

It covers the period from 2022-01-01 to 2023-07-20 and was sampled at 1 Hz. The dataset is intended for environmental and operational analysis.

The 15 SCADA channels contain time-series observations of variables including rotor speed (RPM), generator speed (RPM), stator temperature (°C), wind speed (m/s), converter active power (kW), wind direction relative to the nacelle (degrees), 24 V system supply voltage, pitch angle (degrees), and turbine status.

Source: [Barber et al., Aventa AV-7 SCADA](https://doi.org/10.5281/zenodo.15700928), CC BY 4.0.

# Outline

1. Explore the dataset
2. Define the data splits
3. Define the neural network
4. Train a wind-speed-only model
5. Train a wind-speed-and-pitch-angle model
6. Monitor performance against a reference dataset
7. Investigate hyperparameter importance, with a focus on the learning rate

# Explore the dataset
Before doing any modelling, it is always a good idea to explore the data and make a few plots.

A simple rule to remember is:

Plot the raw data before transforming it.

A complete exploratory data analysis is outside the scope of this workshop, but in general, you should ask yourself:

- Structure: What do I actually have?
Understand the variables, their units, dimensions, sampling, and what each observation represents.
- Quality: Can I trust the data?
Look for missing values, unexpected values, gaps, outliers, or possible measurement issues.
- Intuition: Does the data make sense?
Compare what you observe with what you expect from the physics and your knowledge of the system.
- Relationships: What moves with what?
Explore how variables relate to each other and whether these relationships make physical sense.

The goal is simple: understand what you are modelling before you model it.

## Power Curve

Since we are working with wind turbine SCADA data, plotting the power curve is a natural place to start.
Additionaly, in this data a status of the turbine is available. we can also see how this status evolve

In [ ]:
# File and data handling
from pathlib import Path
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt


# PyTorch: neural networks and data loading
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Lightning: organize the PyTorch training workflow
import lightning as L

# Model evaluation
from sklearn.metrics import mean_absolute_error, r2_score

# Make experiments reproducible
L.seed_everything(42)

# Default style for Matplotlib figures
plt.style.use("seaborn-v0_8-whitegrid")

Let us now connect to the dataset, which comes with this repository.

In [ ]:
path = Path("/workspaces/OpenSD2026_Belgium/notebooks/WS4_AI_for_SHM/data_reg/Aventa_AV7_IET_OST_SCADA_10min.parquet")


data = pd.read_parquet(path).sort_values("Datetime").reset_index(drop=True)
print(f"{len(data):,} ten-minute windows")
print(f"{data.Datetime.min()}  →  {data.Datetime.max()}")
print(f" The columns are: {', '.join(data.columns)}")

Next step is to make some plots.

In [ ]:

labels = {9: "standby", 10: "power operation", 13: "alarm / fault"}
colors = {9: "#1f77b4", 10: "#2ca02c", 13: "#d68427"}

fig, ax = plt.subplot_mosaic(
    [["power", "pitch"],
    ["status", "status"],], figsize=(14, 8),)

for state in labels:
    subset = data[data.StatusAnlage == state]
    ax["power"].scatter(subset['WindSpeed_mean'], subset['PowerOutput_mean'], s=4, alpha=.2, color=colors[state], label=labels[state])
    ax['pitch'].scatter(subset['WindSpeed_mean'], subset['PitchDeg_mean'], s=4, alpha=.2, color=colors[state], label=labels[state])
ax["power"].set(xlabel="Wind speed [m/s]", ylabel="Power output [kW]")
ax["power"].legend()
ax['pitch'].set(xlabel="Wind speed [m/s]", ylabel="Pitch angle [deg]")
ax['pitch'].legend()

timeline = data.set_index("Datetime").groupby([pd.Grouper(freq="7D"), "StatusAnlage"]).size().unstack(fill_value=0)
timeline = timeline.div(timeline.sum(axis=1), axis=0)

ax["status"].stackplot(timeline.index, *[timeline[state] for state in labels], colors=colors.values(), labels=labels.values())
ax["status"].set(xlabel="Time", ylabel="Weekly proportion", ylim=(0, 1))
ax["status"].legend(ncol=3)

plt.tight_layout()
plt.show()

The [Zenodo repository](https://zenodo.org/records/15700928/preview/turbine_status_mapping.json?include_deleted=0) from which the dataset is loaded provides a description of the 13 turbine operating states.

In this notebook, we focus on only three of them:

- **State 9, Standby:** The turbine is in standby mode.
- **State 10, Power operation:** The turbine is actively producing power. Up to the nominal rotor speed of 66 rpm, the blades remain at their basic pitch angle (14°). At higher wind speeds, the pitch angle is increased to regulate the rotor speed and limit power production.
- **State 13, Alarm / fault:** The turbine is in an alarm or fault condition.

At the same wind speed, the controller may permit production, hold the machine in standby, or stop it because of an alarm. No amount of model capacity can infer information that is absent from the inputs.

> **Pause and predict:** If we train one wind-only model on all three states, why would the model fail and what would be the output of the model on that windspeed ?

# Before we start

Before building our first neural network, let's briefly introduce what we are going to use and how the training process works.

There are many libraries available to build and train neural networks. In this notebook, we will use [PyTorch](https://pytorch.org/), one of the most widely used deep-learning libraries in Python.

Our model will be an **Artificial Neural Network (ANN)**. An ANN is composed of layers of interconnected units called **neurons**. The first layer receives the input variables, the **hidden layers** learn transformations of these variables, and the final layer produces the prediction.

<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/4/46/Colored_neural_network.svg" width="450">
</p>

*Example of a feed-forward artificial neural network. Each connection has a weight that is learned during training.*

In our case, the inputs will be turbine measurements such as **wind speed** and optionaly **blade pitch angle**, and the output will be the predicted **power production**.

## How do we train it?

PyTorch gives us all the building blocks required to train a neural network. To keep the training code organized, we will use [Lightning](https://lightning.ai/), which provides a convenient structure around PyTorch.

The recipe is roughly:

1. A **Dataset** defines how we access individual observations.
2. A **DataLoader** groups these observations into small **batches**.
3. A **DataModule** organizes the training, validation, and test data.
4. A **neural network** takes the input variables and makes a prediction.
5. A **loss function** measures the difference between the prediction and the true value.
6. An **optimizer** updates the weights of the network to reduce this loss.

During training, the network does not process the entire dataset at once. Instead, it processes one **batch** at a time. After each batch, the loss is computed and the network weights are updated.

Once the network has seen every batch in the training dataset, it has completed one **epoch**. Training usually consists of several epochs, allowing the network to progressively adjust its weights and improve its predictions.

In the following sections, we will build these components one by one, starting with the data.

# Data Module

The **Data Module** is responsible for preparing the data and creating the DataLoaders used during training, validation, and testing.

## Status split

We will first train the model using **all operating states** and only the wind speed as input. This will show us where a simple power curve model fails when different turbine operating conditions are mixed together.

We will then add the **blade pitch angle** as an additional predictor. The pitch angle provides information about the turbine controller and should help the model explain changes in power production that cannot be explained by wind speed alone.

In a second experiment, we will remove **status 13 (alarm / fault)** from the training data. The model will therefore learn only from normal operating conditions. We can then evaluate it on status 13 and investigate whether faults result in larger prediction errors.

## Splitting the data into train, validation, and test sets

For time series, we should preserve the chronological order of the observations. Randomly splitting the data could allow information from the future to leak into the training set.

We therefore divide the data chronologically into three consecutive parts:

- **Training set:** used to learn the parameters of the neural network.
- **Validation set:** used during training to evaluate the model on unseen data.
- **Test set:** kept aside for the final evaluation.
<p align="center">
  <img src="../../assets/train_validate_test.png" alt="train validation test">
</p>
The important rule is simple: **the model should not learn from the future.**

In [ ]:
VAL_START = pd.Timestamp("2022-10-15")
TEST_START = pd.Timestamp("2022-12-01")
used = ["WindSpeed_mean", "PitchDeg_mean", "PowerOutput_mean", "StatusAnlage"]
data = data.dropna(subset=used).copy()

data["split"] = "train"
data.loc[data.Datetime >= VAL_START, "split"] = "val"
data.loc[data.Datetime >= TEST_START, "split"] = "test"

data.groupby("split").Datetime.agg(start="min", end="max", rows="size")

In [ ]:
class DictDataset(Dataset):
    """PyTorch Dataset that stores DataFrame columns as tensors.

    Each call to dataset[index] returns one observation as a dictionary
    mapping column names to their corresponding tensor values.
    """

    def __init__(self, frame, columns):
        # Convert the selected DataFrame columns to PyTorch tensors
        self.data = {
            c: torch.tensor(frame[c].to_numpy()).float()
            for c in columns
        }

    def __len__(self):
        # Number of observations
        return len(next(iter(self.data.values())))

    def __getitem__(self, index):
        # Retrieve one observation
        return {c: values[index] for c, values in self.data.items()}

> **Try it yourself:** Create a `DictDataset` from the training data and inspect it.  
> Try `len(dataset)` and `dataset[0]`. What does `__getitem__` return?

In [ ]:
class TurbineDataModule(L.LightningDataModule):
    """Organize the turbine data and create DataLoaders for each data split.

    The DataModule optionally filters the data, creates a DictDataset for
    the train, validation, and test sets, and groups observations into batches.
    """

    def __init__(self, data, columns, query=None, batch_size=1024):
        super().__init__()
        self.data, self.columns = data, columns
        self.query, self.batch_size = query, batch_size

    def setup(self, stage=None):
        # Optionally filter the data before creating the datasets
        frame = self.data.query(self.query) if self.query else self.data

        # Create one Dataset for each chronological split
        self.sets = {
            split: DictDataset(frame.query("split == @split"), self.columns)
            for split in ["train", "val", "test"]
        }

    def loader(self, split, shuffle=False):
        # Group individual observations into batches
        return DataLoader(self.sets[split], batch_size=self.batch_size,
                          shuffle=shuffle, drop_last=shuffle)

    def train_dataloader(self): return self.loader("train", True)
    def val_dataloader(self): return self.loader("val")
    def test_dataloader(self): return self.loader("test")

> **Try it yourself:** Create a `TurbineDataModule` and call `setup()`. Inspect `datamodule.sets` and try `len(datamodule.sets["train"])`. Then get one batch with `next(iter(datamodule.train_dataloader()))`. How is a **batch** different from the single observation returned by `DictDataset`?

In [ ]:
columns = ["WindSpeed_mean", "PitchDeg_mean", "PowerOutput_mean"]
all_dm = TurbineDataModule(data, columns)
healthy_dm = TurbineDataModule(data, columns, query="StatusAnlage == 10")

all_dm.setup()
batch = next(iter(all_dm.train_dataloader()))
{k: value.shape for k, value in batch.items()}

Lightning Module to define the model and fitting logic

The network is a sequence of linear maps and ReLU activations. `hidden_units` controls the experiment: `(32, 32)` means two hidden layers; `(64,)` means one.

Batch normalization learns input statistics on training batches. The last layer is linear because power is a regression target.

In [ ]:
class PowerCurveNN(L.LightningModule):
    """Neural network for predicting one turbine variable from a set of inputs.

    The LightningModule defines both the neural network architecture and
    how the model is trained and validated.
    """

    def __init__(self, input_columns, output_name: str, hidden_units=(32, 32), lr=1e-3, l2_weight=1e-4, loss_fn=nn.MSELoss(), activation_fn=nn.ReLU()):
        super().__init__()
        self.save_hyperparameters()

        # Build the network from the requested number of hidden units
        widths = (len(input_columns), *hidden_units)
        layers = [nn.BatchNorm1d(widths[0])]

        for n_in, n_out in zip(widths[:-1], widths[1:]):
            layers += [nn.Linear(n_in, n_out), self.hparams.activation_fn]

        # Final layer returns a single value for the regression
        self.network = nn.Sequential(*layers, nn.Linear(widths[-1], 1))
        self.loss = loss_fn

    def forward(self, x):
        # Define how inputs pass through the neural network
        return self.network(x)

    def shared_step(self, batch, stage):
        # Assemble the selected input variables into [batch, features]
        x = torch.stack([batch[c] for c in self.hparams.input_columns], dim=1)
        y = batch[self.hparams.output_name].unsqueeze(1)

        # Predict and compare with the true output
        loss = self.loss(self(x), y)

        self.log(f"{stage}_loss", loss, on_step=True, on_epoch=True, batch_size=len(y))
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "val")

    def configure_optimizers(self):
        # Adam updates the network weights; weight_decay adds L2 regularization
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.l2_weight)

The following cell defines a function that trains a model to map `input_columns` (can be many input variable) to `output_name` one variable using a provided `datamodule`

In [ ]:
from lightning.pytorch.loggers import CSVLogger

def fit_model(input_columns, output_name, name, datamodule, loss_fn=nn.MSELoss(), activation_fn=nn.ReLU(), l2_weight=1e-4):
    """Create, train, and evaluate a PowerCurveNN.

    Parameters
    ----------
    input_columns : tuple[str]
        Variables given to the neural network as inputs,
        for example ("WindSpeed_mean", "PitchDeg_mean").

    output_name : str
        Variable that the neural network should predict,
        for example "PowerOutput_mean".

    name : str
        Name of the experiment. Used by CSVLogger to create
        a separate directory for the training logs.

    datamodule : TurbineDataModule
        Provides the training and validation DataLoaders.

    loss_fn : nn.Module
        Loss function used to measure the prediction error.
        The default is Mean Squared Error (MSE).

    activation_fn : nn.Module
        Activation function used between the hidden layers.
        The default is ReLU.

    l2_weight : float
        Strength of the L2 regularization. Larger values penalize
        large network weights more strongly.

    Returns
    -------
    model : PowerCurveNN
        The trained neural network.

    logger : CSVLogger
        Logger containing the metrics recorded during training.
    """

    # Create the neural network
    model = PowerCurveNN(input_columns, output_name, loss_fn=loss_fn, activation_fn=activation_fn, l2_weight=l2_weight)

    # Store training metrics in logs/<name>/
    logger = CSVLogger("logs", name=name)

    # Trainer controls the training loop
    trainer = L.Trainer(max_epochs=50, logger=logger, enable_checkpointing=False)

    # Train using the train and validation DataLoaders
    trainer.fit(model, datamodule=datamodule)

    # Read the losses recorded by the logger
    metrics = pd.read_csv(logger.log_dir + "/metrics.csv")
    train_loss = metrics.groupby("epoch")["train_loss_epoch"].mean()
    val_loss = metrics.groupby("epoch")["val_loss_epoch"].mean()

    # Plot how the loss evolves during training
    plt.figure(figsize=(8, 4))
    plt.plot(train_loss, label="Train")
    plt.plot(val_loss, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    return model, logger


def predict(model, frame, columns):
    """Predict the output of a trained model.

    Parameters
    ----------
    model : PowerCurveNN
        Trained neural network.

    frame : pd.DataFrame
        Data containing the observations to predict.

    columns : tuple[str]
        Columns from `frame` used as model inputs.

    Returns
    -------
    np.ndarray
        One prediction for each row of the DataFrame.
    """

    # Convert the input variables to a PyTorch tensor
    x = torch.tensor(frame[list(columns)].to_numpy()).float().to(model.device)

    # Switch from training to evaluation mode
    model.eval()

    # We do not need gradients when making predictions
    with torch.no_grad():
        return model(x).cpu().numpy().ravel()

Lightning does not replace PyTorch. It organizes the steps the trainer will call. The official [LightningModule guide](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html) makes the same distinction: computation stays explicit; repeated loop mechanics are automated.

## Experiment 1: Train on all states

In [ ]:
all_dm = TurbineDataModule(data, columns)

In [ ]:
wind_model, wind_log = fit_model(("WindSpeed_mean",), "PowerOutput_mean", "all_states_wind", all_dm)

<details>
<summary><b>Why is the validation loss lower than the training loss?</b></summary>

A lower validation loss does not necessarily mean that something is wrong.

In our case, there are two possible explanations:

1. **The training and validation sets are different.**  
   Since we use a chronological split, the two sets may contain different operating conditions. The validation period may simply be easier for the model to predict.

2. **Batch Normalization behaves differently during training and validation.**  
   During training, `BatchNorm` uses the statistics of the current batch. During validation, it uses the running statistics learned during training, which can result in more stable predictions.

More importantly, look at the **evolution** of the two curves. Both losses decrease during the first epochs and then stabilize. We do not observe the typical sign of overfitting, where the training loss continues to decrease while the validation loss starts to increase.

**Try it yourself:** Compare the distributions and power curves of the training and validation sets. Is the validation period actually easier to predict?

</details>

In [ ]:
def predict(model, frame, columns):
    x = torch.tensor(frame[list(columns)].to_numpy()).float().to(model.device)
    model.eval()
    with torch.no_grad():
        return model(x).cpu().numpy().ravel()

val_data = data.query("split == 'val'").copy()
val_data["wind_error"] = (
    val_data['PowerOutput_mean'] - predict(wind_model, val_data, ("WindSpeed_mean",))
)

In [ ]:
# Let's plot the power curve learned by the model. 
# We will use the validation set for this.
val_data = data.query("split == 'val'").copy()
val_data["wind_error"] = (
    val_data.PowerOutput_mean - predict(wind_model, val_data, ("WindSpeed_mean",))
)
fig, ax = plt.subplots(ncols=2, figsize=(12, 4))

ax[0].scatter(val_data.PitchDeg_mean, val_data.wind_error, s=4, alpha=.2)
ax[0].axhline(0, color="black", ls="--")
ax[0].set(xlabel="Pitch angle [degree]", ylabel="Wind-only residual [kW]")

val_data = val_data.sort_values("WindSpeed_mean")
wind_prediction = val_data.PowerOutput_mean - val_data.wind_error

ax[1].scatter(val_data.WindSpeed_mean, val_data.PowerOutput_mean, s=4, alpha=.2)
ax[1].plot(val_data.WindSpeed_mean, wind_prediction, color="red")
ax[1].set(xlabel="Wind speed [m/s]", ylabel="Power output [kW]")

plt.tight_layout()
plt.show()

The wind-only model cannot represent the complete power curve because **wind speed alone does not uniquely determine power output**.

The left figure helps explain why. The residual is strongly related to the **pitch angle**, meaning that pitch contains information that is missing from the wind-only model.

In regions where the same wind speed corresponds to different power outputs, the regression model cannot know which operating condition the turbine is in. With an MSE loss, its best solution tends toward the **conditional mean** of the possible outputs. This explains why the learned curve passes between the different operating regimes rather than following each of them.

> **Next step:** Add the pitch angle as an input and see whether providing information about the turbine controller improves the prediction.

In [ ]:
pitch_model, pitch_log = fit_model(
    ("WindSpeed_mean", "PitchDeg_mean"),'PowerOutput_mean', "all_states_wind_pitch", all_dm,loss_fn=nn.SmoothL1Loss()
)

Adding the **pitch angle** as an input strongly reduces both the training and validation losses. Additionaly, The two curves are now also much closer to each other (the distribution of the two dataset is close)

This supports our previous observation: **wind speed alone was missing important information about the turbine operating condition**. The pitch angle provides information about the controller response, allowing the network to distinguish operating conditions that previously looked identical from the wind speed alone.

The smaller gap between training and validation loss also suggests that the relationship learned from the training period transfers well to the validation period.

> The improvement does not come from making the neural network more complex. We kept the same architecture and simply provided it with a more informative input variable. **Better features can matter more than a more complex model.** This may seem obvious in this case but in other scenarios this could an enabler (..discuss this with me)

In [ ]:
val_data = data.query("split == 'val'").copy()
val_data["prediction"] = predict(pitch_model, val_data, ("WindSpeed_mean", "PitchDeg_mean"))
val_data["residual"] = val_data.PowerOutput_mean - val_data.prediction

fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122, projection="3d")

ax1.scatter(val_data.PitchDeg_mean, val_data.residual, s=4, alpha=.2)
ax1.axhline(0, color="black", ls="--")
ax1.set(xlabel="Pitch angle [degree]", ylabel="Residual [kW]")

ax2.scatter(val_data.WindSpeed_mean, val_data.PitchDeg_mean, val_data.PowerOutput_mean, s=4, alpha=.1, label="Actual")
ax2.scatter(val_data.WindSpeed_mean, val_data.PitchDeg_mean, val_data.prediction, s=4, alpha=.3, color="red", label="Predicted")
ax2.set(xlabel="Wind speed [m/s]", ylabel="Pitch angle [degree]", zlabel="Power [kW]")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_data = data.query("split == 'test'").copy()
y_true = test_data['PowerOutput_mean'].to_numpy()
wind_pred = predict(wind_model, test_data, ("WindSpeed_mean",))
pitch_pred = predict(pitch_model, test_data, ("WindSpeed_mean", "PitchDeg_mean"))

results = pd.DataFrame({
    "MAE": [mean_absolute_error(y_true, wind_pred),
            mean_absolute_error(y_true, pitch_pred)],
    "R2": [r2_score(y_true, wind_pred), r2_score(y_true, pitch_pred)],
}, index=["wind", "wind + pitch"])
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
limits = [y_true.min(), y_true.max()]
for ax, prediction, title in zip(
    axes, [wind_pred, pitch_pred], results.index
):
    ax.scatter(y_true, prediction, s=4, alpha=.2)
    ax.plot(limits, limits, "k--")
    ax.set(title=title, xlabel="Actual power", ylabel="Predicted power")
plt.tight_layout()

If pitch improves the all-state test score, it helps describe controller-mediated output. Wind speed describes the available resource; pitch describes the controller's response. Pitch tells the model whether the blades are capturing or limiting aerodynamic power, helping separate operating conditions that wind alone cannot distinguish.

## Experiment 2: A separate healthy reference
For condition monitoring, fit a wind-only model on state 10. This estimates `power | wind, normal operation` and produces one interpretable curve.

In [ ]:
healthy_dm = TurbineDataModule(data, columns, query="StatusAnlage == 10")
healthy_model, healthy_log = fit_model(
    ("WindSpeed_mean",),'PowerOutput_mean', "healthy_wind", healthy_dm
)
healthy_test = data.query("split == 'test' and StatusAnlage == 10").copy()
healthy_pred = predict(healthy_model, healthy_test, ("WindSpeed_mean",))
order = np.argsort(healthy_test.WindSpeed_mean.to_numpy())

plt.scatter(healthy_test['WindSpeed_mean'], healthy_test['PowerOutput_mean'], s=3, alpha=.1)
plt.plot(healthy_test['WindSpeed_mean'].iloc[order], healthy_pred[order], color="red")
plt.xlabel("Wind speed [m/s]")
plt.ylabel("Power output [kW]")
plt.show()

> play with the model hyperrameter to improve the power fit curve  (improve extrapolation ) try tanh activation function 

> add A ReLU layer at the end of the model to avoid negative values


In [ ]:
healthy_val = data.query("split == 'val' and StatusAnlage == 10").copy()
val_prediction = predict(healthy_model, healthy_val, ("WindSpeed_mean",))
val_error = np.abs(healthy_val['PowerOutput_mean'].to_numpy() - val_prediction)
ERROR_QUANTILE = .99  # Try .95 and .999
threshold = np.quantile(val_error, ERROR_QUANTILE)
threshold

In [ ]:
monitor = data.query("split == 'test'").sort_values("Datetime").copy()
monitor["expected"] = predict(healthy_model, monitor, ("WindSpeed_mean",))
monitor["error"] = (monitor.PowerOutput_mean - monitor.expected).abs()
monitor["trigger"] = monitor.error > threshold

In [ ]:
alarm = monitor.StatusAnlage.eq(13)

plt.figure(figsize=(16,5))
plt.plot(monitor.Datetime, monitor.error, lw=.6, alpha=.45, label="absolute residual")
plt.scatter(monitor.loc[alarm, "Datetime"], monitor.loc[alarm, "error"],
            s=4, c="red", label="state 13")

plt.axhline(threshold, ls="--", label="threshold")

plt.xlabel("Time")
plt.ylabel("Absolute power residual [kW]")
plt.legend()
plt.show()

In [ ]:
monitor.groupby("StatusAnlage").agg(
    windows=("error", "size"),
    median_error=("error", "median"),
    trigger_rate=("trigger", "mean"),
)

### Is deep learning necessary here?

No. For a small tabular problem with one or two inputs, a decision tree, boosted trees, a spline, or a standard power-curve model may perform as well while being faster and easier to interpret. We use a neural network here to learn the PyTorch/Lightning workflow and to make feature and architecture experiments easy. Deep learning becomes more compelling when many sensors, turbines, or more complex temporal relationships are modelled together.

## What to remember

- Wind speed does not identify controller state.
- A power curve answers `power | wind, normal operation`, not power under every condition.
- Keep train, validation, and future test roles separate.
- A batch is `[observations, features]`.
- Lightning helps organizes PyTorch experiments

# Investigate hyperparameter importance, with a focus on the learning rate
Let's investigate the impact of different hyperparamteres on the model's performance, particularly focusing on the learning rate. We will train multiple models with varying learning rates and compare their validation losses to determine the optimal learning rate for our neural network. 

In [ ]:
# Hyperparameter study: change one choice at a time from the baseline
# Run this at your own risk may crash in codespace and should be run locally
if False: 
    experiments = {
        "baseline": {"hidden_units": (32, 32), "loss_fn": nn.MSELoss(), "l2_weight": 1e-4, "lr": 1e-3},
        "low LR": {"hidden_units": (32, 32), "loss_fn": nn.MSELoss(), "l2_weight": 1e-4, "lr": 1e-4},
        "mid LR": {"hidden_units": (32, 32), "loss_fn": nn.MSELoss(), "l2_weight": 1e-4, "lr": 1e-3},
        "high LR": {"hidden_units": (32, 32), "loss_fn": nn.MSELoss(), "l2_weight": 1e-4, "lr": 1e-2},
        "SmoothL1 loss": {"hidden_units": (32, 32), "loss_fn": nn.SmoothL1Loss(), "l2_weight": 1e-4, "lr": 1e-3},
        "small model": {"hidden_units": (8,), "loss_fn": nn.MSELoss(), "l2_weight": 1e-4, "lr": 1e-3},
        "large model": {"hidden_units": (64, 64, 32), "loss_fn": nn.MSELoss(), "l2_weight": 1e-4, "lr": 1e-3},
        "no L2": {"hidden_units": (32, 32), "loss_fn": nn.MSELoss(), "l2_weight": 0, "lr": 1e-3},
        "strong L2": {"hidden_units": (32, 32), "loss_fn": nn.MSELoss(), "l2_weight": 1e-1, "lr": 1e-3},
    }

    scores = []
    validation = data.query("split == 'val'")

    for name, settings in experiments.items():
        L.seed_everything(42)
        trial_model = PowerCurveNN(("WindSpeed_mean", "PitchDeg_mean"),'PowerOutput_mean', **settings)
        trial_trainer = L.Trainer(max_epochs=30, logger=False, enable_checkpointing=False, enable_progress_bar=False)
        trial_trainer.fit(trial_model, datamodule=all_dm)
        prediction = predict(trial_model, validation, ("WindSpeed_mean", "PitchDeg_mean"))
        scores.append({"experiment": name, "MAE": mean_absolute_error(validation.PowerOutput_mean, prediction), "R2": r2_score(validation.PowerOutput_mean, prediction)})

    hyperparameter_results = pd.DataFrame(scores).set_index("experiment")
    display(hyperparameter_results.sort_values("MAE"))
    hyperparameter_results.plot.bar(subplots=True, figsize=(10, 5), rot=30, legend=False)
    plt.tight_layout()
    plt.show()

### Observations

The baseline already performs well. Changing the learning rate has the largest effect: a low learning rate converges too slowly within 30 epochs, while a high learning rate makes optimization unstable. The middle learning rate gives performance close to the baseline.

The small model underfits, whereas the larger model gives only a small improvement. Smooth L1 performs similarly to MSE, suggesting that outliers are not dominating this validation set. L2 regularization has little effect at its default strength, but strong L2 slightly reduces performance.

These conclusions are based on one split and one random seed, so small differences should not be overinterpreted.